# Salud Chilecito - Demo DAO y Plataforma

Este notebook demuestra la capa DAO (Data Access Object) de Salud Chilecito, mostrando la estructura de las tablas de la base de datos y las operaciones CRUD disponibles.

**Formas de uso del proyecto:**

- `http://localhost:8000`: plataforma gráfica
- `src/dao`: capa DAO para Oracle
- `sql/`: scripts Oracle con tablespaces, usuarios, esquema, índices, seed y validaciones

## 1. Preparación

Ejecutar el notebook desde la raíz del repositorio:

```bash
Windows: .\scripts\windows\04_abrir_notebook.ps1
Ubuntu: bash scripts/ubuntu/04_abrir_notebook.sh
```

Si todavía no se instalaron dependencias:

```bash
python -m venv .venv
.venv\Scripts\activate        # Windows
source .venv/bin/activate      # Ubuntu
pip install -r requirements.txt
```

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print("Repositorio:", ROOT)
print("Existe src/dao:", (ROOT / "src" / "dao").exists())
print("Existe sql/:", (ROOT / "sql").exists())

## 2. Estructura de la Base de Datos

El sistema utiliza Oracle como base de datos con las siguientes tablas:

In [ ]:
# Definición de las tablas de la base de datos

tablas = {
    "CENTROS": """
CREATE TABLE centros (
    id NUMBER PRIMARY KEY,
    nombre VARCHAR2(100) NOT NULL,
    slug VARCHAR2(100),
    direccion VARCHAR2(200),
    distrito VARCHAR2(100),
    telefono VARCHAR2(50),
    tipo VARCHAR2(20) CHECK (tipo IN ('HOSPITAL', 'CLINICA', 'CENTRO_SALUD'))
);
""",
    
    "ESPECIALIDADES": """
CREATE TABLE especialidades (
    id NUMBER PRIMARY KEY,
    nombre VARCHAR2(100) NOT NULL,
    descripcion VARCHAR2(500)
);
""",
    
    "MEDICOS": """
CREATE TABLE medicos (
    id NUMBER PRIMARY KEY,
    nombre VARCHAR2(100) NOT NULL,
    especialidad_id NUMBER REFERENCES especialidades(id),
    centro_id NUMBER REFERENCES centros(id),
    telefono VARCHAR2(50)
);
""",
    
    "PACIENTES": """
CREATE TABLE pacientes (
    id NUMBER PRIMARY KEY,
    dni VARCHAR2(20) UNIQUE NOT NULL,
    nombre VARCHAR2(100) NOT NULL,
    telefono VARCHAR2(50),
    obra_social VARCHAR2(100),
    distrito VARCHAR2(100),
    centro_id NUMBER REFERENCES centros(id)
);
""",
    
    "TURNOS": """
CREATE TABLE turnos (
    id NUMBER PRIMARY KEY,
    paciente_id NUMBER REFERENCES pacientes(id),
    medico_id NUMBER REFERENCES medicos(id),
    centro_id NUMBER REFERENCES centros(id),
    fecha DATE NOT NULL,
    hora VARCHAR2(5) NOT NULL,
    estado VARCHAR2(20) CHECK (estado IN ('PENDIENTE', 'CONFIRMADO', 'ATENDIDO', 'CANCELADO', 'AUSENTE')),
    precio NUMBER,
    motivo VARCHAR2(500),
    origen VARCHAR2(20) DEFAULT 'VIRTUAL',
    fecha_creacion TIMESTAMP DEFAULT SYSTIMESTAMP
);
""",
    
    "AGENDAS": """
CREATE TABLE agendas (
    id NUMBER PRIMARY KEY,
    medico_id NUMBER REFERENCES medicos(id),
    dia_semana VARCHAR2(20) NOT NULL,
    hora_inicio VARCHAR2(5) NOT NULL,
    hora_fin VARCHAR2(5) NOT NULL,
    duracion_minutos NUMBER NOT NULL,
    cupo_diario NUMBER NOT NULL
);
""",
    
    "DOCUMENTOS": """
CREATE TABLE documentos (
    id NUMBER PRIMARY KEY,
    paciente_id NUMBER REFERENCES pacientes(id),
    tipo VARCHAR2(50),
    nombre_archivo VARCHAR2(200),
    ruta VARCHAR2(500),
    mime_type VARCHAR2(100),
    tamano_bytes NUMBER
);
""",
    
    "SINTOMAS": """
CREATE TABLE sintomas (
    id NUMBER PRIMARY KEY,
    descripcion VARCHAR2(200) NOT NULL,
    especialidad_id NUMBER REFERENCES especialidades(id),
    prioridad VARCHAR2(20) CHECK (prioridad IN ('ALTA', 'MEDIA', 'BAJA'))
);
""",
    
    "CONFIGURACION_HOSPITAL": """
CREATE TABLE configuracion_hospital (
    id NUMBER PRIMARY KEY,
    nombre_hospital VARCHAR2(100),
    centro_principal NUMBER REFERENCES centros(id),
    mensaje_bienvenida VARCHAR2(500),
    color_primario VARCHAR2(20),
    color_secundario VARCHAR2(20),
    tiempo_cancelacion_horas NUMBER DEFAULT 24
);
""",
    
    "TIPOS_CONSULTA": """
CREATE TABLE tipos_consulta (
    id NUMBER PRIMARY KEY,
    nombre VARCHAR2(50) NOT NULL,
    descripcion VARCHAR2(200)
);
""",
    
    "PRECIOS_ESPECIALIDAD": """
CREATE TABLE precios_especialidad (
    id NUMBER PRIMARY KEY,
    centro_id NUMBER REFERENCES centros(id),
    especialidad_id NUMBER REFERENCES especialidades(id),
    tipo_consulta_id NUMBER REFERENCES tipos_consulta(id),
    precio_estimado NUMBER,
    precio_minimo NUMBER,
    precio_maximo NUMBER,
    duracion_minutos NUMBER
);
"""
}

for nombre, sql in tablas.items():
    print(f"\n### {nombre}")
    print(sql)

In [ ]:
## 3. Capa DAO - Data Access Object

Los DAO son la interfaz formal entre Python y Oracle. Las consultas quedan agrupadas en clases específicas para cada entidad.

# Importar todos los DAOs disponibles
from src.dao import (
    CentroDAO,
    PacienteDAO,
    TurnoDAO,
    MedicoDAO,
    EspecialidadDAO,
    AgendaDAO,
    HistorialDAO,
    APISubscriptionDAO
)

print("DAOs disponibles:")
print("- CentroDAO")
print("- PacienteDAO")
print("- TurnoDAO")
print("- MedicoDAO")
print("- EspecialidadDAO")
print("- AgendaDAO")
print("- HistorialDAO")
print("- APISubscriptionDAO")

In [ ]:
## 4. Ejemplos de Operaciones CRUD

A continuación se muestran ejemplos funcionales de las operaciones CRUD para cada DAO.

### 4.1 CentroDAO - Operaciones CRUD

In [ ]:
# Ejemplo de uso de CentroDAO
try:
    centro_dao = CentroDAO()
    
    # READ - Listar todos los centros
    centros = centro_dao.listar()
    print("Centros disponibles:")
    for centro in centros:
        print(f"  - {centro['nombre']} ({centro['tipo']}) - {centro['distrito']}")
    
    # READ - Filtrar por distrito
    centros_chilecito = centro_dao.listar(distrito="Chilecito")
    print(f"\nCentros en Chilecito: {len(centros_chilecito)}")
    
except Exception as e:
    print(f"Error al usar CentroDAO: {e}")
    print("Usando modo demo con datos JSON")

### 4.2 PacienteDAO - Operaciones CRUD

In [ ]:
# Ejemplo de uso de PacienteDAO
try:
    paciente_dao = PacienteDAO()
    
    # READ - Listar todos los pacientes
    pacientes = paciente_dao.listar()
    print(f"Pacientes registrados: {len(pacientes)}")
    for paciente in pacientes:
        print(f"  - {paciente['nombre']} (DNI: {paciente['dni']}) - {paciente['distrito']}")
    
    # READ - Buscar por DNI
    paciente = paciente_dao.buscar_por_dni("12345678")
    if paciente:
        print(f"\nPaciente encontrado: {paciente['nombre']}")
    
except Exception as e:
    print(f"Error al usar PacienteDAO: {e}")
    print("Usando modo demo con datos JSON")

In [ ]:
### 4.3 MedicoDAO - Operaciones CRUD

In [ ]:
# Ejemplo de uso de MedicoDAO
try:
    medico_dao = MedicoDAO()
    
    # READ - Listar todos los médicos
    medicos = medico_dao.listar()
    print(f"Médicos registrados: {len(medicos)}")
    for medico in medicos:
        print(f"  - Dr. {medico['nombre']} - {medico.get('especialidad_nombre', 'N/A')}")
    
    # READ - Filtrar por especialidad
    medicos_cardiologia = medico_dao.listar_por_especialidad(1)
    print(f"\nMédicos de Cardiología: {len(medicos_cardiologia)}")
    
except Exception as e:
    print(f"Error al usar MedicoDAO: {e}")
    print("Usando modo demo con datos JSON")

### 4.4 EspecialidadDAO - Operaciones CRUD

# Ejemplo de uso de EspecialidadDAO
try:
    especialidad_dao = EspecialidadDAO()
    
    # READ - Listar todas las especialidades
    especialidades = especialidad_dao.listar()
    print(f"Especialidades disponibles: {len(especialidades)}")
    for esp in especialidades:
        print(f"  - {esp['nombre']}")
    
except Exception as e:
    print(f"Error al usar EspecialidadDAO: {e}")
    print("Usando modo demo con datos JSON")

### 4.5 TurnoDAO - Operaciones CRUD

### 4.6 AgendaDAO - Operaciones CRUD

In [ ]:
# Ejemplo de uso de AgendaDAO
try:
    agenda_dao = AgendaDAO()
    
    # READ - Listar todas las agendas
    agendas = agenda_dao.listar()
    print(f"Agendas registradas: {len(agendas)}")
    for agenda in agendas:
        print(f"  - Médico #{agenda['medico_id']}: {agenda['dia_semana']} {agenda['hora_inicio']}-{agenda['hora_fin']}")
    
except Exception as e:
    print(f"Error al usar AgendaDAO: {e}")
    print("Usando modo demo con datos JSON")

## 5. Resumen del Sistema

In [ ]:
# Resumen de la arquitectura del sistema

print("=== ARQUITECTURA DEL SISTEMA ===")
print("\nCapas del sistema:")
print("1. Presentación: Web (JavaScript) + API REST")
print("2. Lógica de negocio: Store (JSON/Oracle)")
print("3. Acceso a datos: DAO (Data Access Object)")
print("4. Base de datos: Oracle")

print("\nTablas principales:")
for nombre in tablas.keys():
    print(f"  - {nombre}")

print("\nDAOs disponibles:")
print("  - CentroDAO")
print("  - PacienteDAO")
print("  - TurnoDAO")
print("  - MedicoDAO")
print("  - EspecialidadDAO")
print("  - AgendaDAO")
print("  - HistorialDAO")
print("  - APISubscriptionDAO")

## 6. Comandos de Ejecución

In [ ]:
# Comandos para ejecutar el proyecto

print("=== COMANDOS DE EJECUCIÓN ===")
print("\nInstalar dependencias:")
print("python -m venv .venv")
print(".venv\\Scripts\\activate        # Windows")
print("source .venv/bin/activate      # Ubuntu")
print("pip install -r requirements.txt")

print("\nEjecutar servidor:")
print("python -m src.webapp.server")

print("\nEjecutar tests:")
print("python -m pytest -q")

print("\nAbrir notebook:")
print("Windows: .\\scripts\\windows\\04_abrir_notebook.ps1")
print("Ubuntu: bash scripts/ubuntu/04_abrir_notebook.sh")

## 7. URLs de Acceso

In [ ]:
# URLs de acceso a la plataforma

print("=== URLs DE ACCESO ===")
print("\nPlataforma gráfica:")
print("  http://localhost:8000")
print("\nLanding page del hospital:")
print("  http://localhost:8000/landing")
print("\nAPI REST:")
print("  http://localhost:8000/api/centros")
print("  http://localhost:8000/api/pacientes")
print("  http://localhost:8000/api/turnos")
print("  http://localhost:8000/api/medicos")
print("  http://localhost:8000/api/especialidades")

# Ejemplo de uso de TurnoDAO
try:
    turno_dao = TurnoDAO()
    
    # READ - Listar todos los turnos
    turnos = turno_dao.listar()
    print(f"Turnos registrados: {len(turnos)}")
    for turno in turnos:
        print(f"  - Turno #{turno['id']}: {turno['fecha']} {turno['hora']} - {turno['estado']}")
    
    # READ - Filtrar por paciente
    turnos_paciente = turno_dao.listar_por_paciente(1)
    print(f"\nTurnos del paciente #1: {len(turnos_paciente)}")
    
except Exception as e:
    print(f"Error al usar TurnoDAO: {e}")
    print("Usando modo demo con datos JSON")